##### ### The University of Melbourne, School of Computing and Information Systems
# COMP30027 Machine Learning, 2026 Semester 1

## Assignment 1: Income Classification with Naïve Bayes


**Student ID(s):**     1615136

**Student Name:**      Skanda Ramanan


This iPython notebook is a template which you will use for your Assignment 1 submission.

**NOTE: YOU SHOULD ADD YOUR RESULTS, GRAPHS, AND FIGURES FROM YOUR OBSERVATIONS IN THIS FILE TO YOUR REPORT (the PDF file).** Results, figures, etc. which appear in this file but are NOT included in your report will not be marked.

**Adding proper comments to your code is MANDATORY. **

## 1. Supervised model training


In [1]:
import pandas as pd
import numpy as np
from sklearn.naive_bayes import GaussianNB, CategoricalNB
from sklearn.preprocessing import OrdinalEncoder
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# --- Data Loading ---
train_df = pd.read_csv('adult_supervised_train.csv')

print(f"Raw training set shape: {train_df.shape}")
print(f"\nColumn dtypes:\n{train_df.dtypes}")
print(f"\nFirst 3 rows:")
train_df.head(3)

Raw training set shape: (16280, 15)

Column dtypes:
age                int64
workclass         object
fnlwgt             int64
education         object
education-num      int64
marital-status    object
occupation        object
relationship      object
race              object
sex               object
capital-gain       int64
capital-loss       int64
hours-per-week     int64
native-country    object
income            object
dtype: object

First 3 rows:


,age,workclass,fnlwgt,education,education-num,marital-status,occupation,relationship,race,sex,capital-gain,capital-loss,hours-per-week,native-country,income
0,63,Private,219337,7th-8th,4,Married-civ-spouse,Craft-repair,Husband,White,Male,3471,0,45,United-States,<=50K
1,44,State-gov,33658,Masters,14,Married-civ-spouse,Prof-specialty,Husband,White,Male,7688,0,50,United-States,>50K
2,18,Private,157193,12th,8,Never-married,Other-service,Own-child,White,Male,0,0,30,Italy,<=50K


In [2]:
# --- Preprocessing ---

# Replace '?' with NaN and drop rows with any missing values
rows_before = len(train_df) 
train_df.dropna(inplace=True)
rows_after = len(train_df)
print(f"Dropped {rows_before - rows_after} rows with missing values ({rows_before} -> {rows_after})")

# Drop fnlwgt — not listed as a feature in the assignment spec
train_df.drop(columns=['fnlwgt'], inplace=True)

# Define feature groups per the assignment specification
CONTINUOUS_FEATURES = ['age', 'education-num', 'capital-gain', 'capital-loss', 'hours-per-week']
CATEGORICAL_FEATURES = ['workclass', 'education', 'marital-status', 'occupation',
                        'relationship', 'race', 'sex', 'native-country']

# Encode target: <=50K -> 0, >50K -> 1
train_df['income'] = train_df['income'].map({'<=50K': 0, '>50K': 1})

# Separate features and target
X_train_cont = train_df[CONTINUOUS_FEATURES].astype(float)
X_train_cat = train_df[CATEGORICAL_FEATURES]
y_train = train_df['income']

print(f"\nContinuous features: {CONTINUOUS_FEATURES}")
print(f"Categorical features: {CATEGORICAL_FEATURES}")
print(f"\nClass distribution:\n{y_train.value_counts().rename({0: '<=50K', 1: '>50K'})}")

Dropped 1204 rows with missing values (16280 -> 15076)

Continuous features: ['age', 'education-num', 'capital-gain', 'capital-loss', 'hours-per-week']
Categorical features: ['workclass', 'education', 'marital-status', 'occupation', 'relationship', 'race', 'sex', 'native-country']

Class distribution:
income
<=50K    11369
>50K      3707
Name: count, dtype: int64


In [3]:
# --- Train Mixed Naive Bayes ---
# GaussianNB for continuous features, CategoricalNB for categorical features

# 1) Gaussian component — fits mean and variance per class per continuous feature
gnb = GaussianNB()
gnb.fit(X_train_cont, y_train)

# 2) Categorical component — ordinal-encode then fit with Laplace smoothing (alpha=1)
encoder = OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1)
X_train_cat_enc = encoder.fit_transform(X_train_cat).astype(int)

cnb = CategoricalNB(alpha=1.0)
cnb.fit(X_train_cat_enc, y_train)

print("Mixed Naive Bayes trained successfully.")
print(f"  GaussianNB classes: {gnb.classes_}")
print(f"  CategoricalNB classes: {cnb.classes_}")


def predict_mixed_nb(X_cont, X_cat_enc, gnb, cnb):
    """
    Combine GaussianNB and CategoricalNB log-posteriors to produce
    final predictions from the mixed model.
    """
    # Joint log-likelihood + log-prior from each sub-model
    log_prob_gnb = gnb._joint_log_likelihood(X_cont)
    log_prob_cnb = cnb._joint_log_likelihood(X_cat_enc.astype(int))

    # Each already includes log P(c), so subtract one copy of the prior
    # to avoid double-counting: log P(c|x) ∝ log P(c) + Σ log P(xj|c)
    log_prior = np.log(gnb.class_prior_)
    log_posterior = log_prob_gnb + log_prob_cnb - log_prior

    return gnb.classes_[np.argmax(log_posterior, axis=1)], log_posterior


# Verify on training data
y_pred_train, _ = predict_mixed_nb(X_train_cont, X_train_cat_enc, gnb, cnb)
train_acc = accuracy_score(y_train, y_pred_train)
print(f"\nTraining accuracy (sanity check): {train_acc:.4f}")

Mixed Naive Bayes trained successfully.
  GaussianNB classes: [0 1]
  CategoricalNB classes: [0 1]

Training accuracy (sanity check): 0.8289


In [4]:
# --- Q1.1: Prior Probabilities ---

prior_low = gnb.class_prior_[0]   # P(<=50K)
prior_high = gnb.class_prior_[1]  # P(>50K)

print("Prior Probabilities P(c):")
print(f"  P(<=50K) = {prior_low:.4f}")
print(f"  P(>50K)  = {prior_high:.4f}")
print(f"\nClass ratio (<=50K : >50K) = {prior_low/prior_high:.2f} : 1")
print(f"\nThe training set exhibits class imbalance — the <=50K class is "
      f"~{prior_low/prior_high:.1f}x more frequent than >50K.")

Prior Probabilities P(c):
  P(<=50K) = 0.7541
  P(>50K)  = 0.2459

Class ratio (<=50K : >50K) = 3.07 : 1

The training set exhibits class imbalance — the <=50K class is ~3.1x more frequent than >50K.


In [5]:
# --- Q1.2: Continuous Feature Statistics per Class ---

# Extract learned parameters from GaussianNB
means = gnb.theta_       # shape (n_classes, n_features)
variances = gnb.var_     # shape (n_classes, n_features)
stds = np.sqrt(variances)

# Build a summary table
cont_stats = pd.DataFrame({
    'Feature': CONTINUOUS_FEATURES,
    'Mean (<=50K)': means[0],
    'Std (<=50K)': stds[0],
    'Mean (>50K)': means[1],
    'Std (>50K)': stds[1],
})

# Measure class separation via Cohen's d: |μ1 - μ2| / sqrt((σ1² + σ2²) / 2)
cont_stats["Cohen's d"] = np.abs(means[1] - means[0]) / np.sqrt((variances[0] + variances[1]) / 2)
cont_stats = cont_stats.sort_values("Cohen's d", ascending=False)

print("Continuous Feature Statistics by Class:")
print("=" * 90)
print(cont_stats.to_string(index=False, float_format='{:.4f}'.format))
print("\n(Sorted by Cohen's d — larger values indicate greater separation between classes)")

Continuous Feature Statistics by Class:
       Feature  Mean (<=50K)  Std (<=50K)  Mean (>50K)  Std (>50K)  Cohen's d
 education-num        9.6293       2.4478      11.5946      2.3650     0.8166
           age       37.0508      13.7119      43.9385     10.3027     0.5679
hours-per-week       39.4286      11.9108      45.6423     10.3961     0.5558
  capital-gain      157.6660    1017.8632    3607.1467  13616.6230     0.3573
  capital-loss       55.9746     316.0201     202.3620    603.9910     0.3037

(Sorted by Cohen's d — larger values indicate greater separation between classes)


In [6]:
# --- Q1.3: Categorical Feature Probability Ratios ---

# Recover P(xj = v | c) for each categorical feature from the CategoricalNB model.
# cnb.feature_log_prob_ is a list of arrays, one per feature,
# each of shape (n_classes, n_categories_for_that_feature).

ratio_records = []

for feat_idx, feat_name in enumerate(CATEGORICAL_FEATURES):
    # Log-probabilities for this feature: shape (n_classes, n_categories)
    log_probs = cnb.feature_log_prob_[feat_idx]
    probs = np.exp(log_probs)  # convert to probabilities

    # Category labels from the encoder
    categories = encoder.categories_[feat_idx]

    for cat_idx, cat_value in enumerate(categories):
        p_given_low = probs[0, cat_idx]   # P(v | <=50K)
        p_given_high = probs[1, cat_idx]  # P(v | >50K)

        # R for >50K: how much more likely under >50K than <=50K
        r_high = p_given_high / p_given_low
        # R for <=50K: how much more likely under <=50K than >50K
        r_low = p_given_low / p_given_high

        ratio_records.append({
            'Feature': feat_name,
            'Value': cat_value,
            'P(v|<=50K)': p_given_low,
            'P(v|>50K)': p_given_high,
            'R (>50K)': r_high,
            'R (<=50K)': r_low,
        })

ratio_df = pd.DataFrame(ratio_records)

# Top 5 most predictive of >50K (highest R for >50K)
top5_high = ratio_df.nlargest(5, 'R (>50K)')[['Feature', 'Value', 'P(v|<=50K)', 'P(v|>50K)', 'R (>50K)']]
print("Top 5 Category Values Most Predictive of >50K:")
print("=" * 75)
print(top5_high.to_string(index=False, float_format='{:.4f}'.format))

print()

# Top 5 most predictive of <=50K (highest R for <=50K)
top5_low = ratio_df.nlargest(5, 'R (<=50K)')[['Feature', 'Value', 'P(v|<=50K)', 'P(v|>50K)', 'R (<=50K)']]
print("Top 5 Category Values Most Predictive of <=50K:")
print("=" * 75)
print(top5_low.to_string(index=False, float_format='{:.4f}'.format))

Top 5 Category Values Most Predictive of >50K:
       Feature             Value  P(v|<=50K)  P(v|>50K)  R (>50K)
     education       Prof-school      0.0069     0.0545    7.9587
     education         Doctorate      0.0045     0.0320    7.1354
marital-status Married-AF-spouse      0.0004     0.0019    4.2882
     education           Masters      0.0357     0.1268    3.5551
native-country            Taiwan      0.0008     0.0027    3.3832

Top 5 Category Values Most Predictive of <=50K:
     Feature           Value  P(v|<=50K)  P(v|>50K)  R (<=50K)
relationship       Own-child      0.1858     0.0097    19.1680
  occupation Priv-house-serv      0.0064     0.0005    11.9315
  occupation   Other-service      0.1389     0.0188     7.3831
relationship  Other-relative      0.0373     0.0054     6.9201
   education             9th      0.0196     0.0030     6.6294


## 2. Supervised model evaluation

## 3. Extending the model with semi-supervised training

## 4. Supervised model evaluation